In [1]:
from datasets import load_dataset
import pandas as pd
from collections import defaultdict
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import json
import re

import random
from sklearn.metrics.pairwise import cosine_similarity
import time
import psutil
import os
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
from collections import Counter

# Bước chuẩn bị

### Load dataset train/test/valid

In [2]:
# Load dataset
dataset = load_dataset("yammdd/vietnamese-error-correction-corpus")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'target'],
        num_rows: 56445
    })
    validation: Dataset({
        features: ['input', 'target'],
        num_rows: 7056
    })
    test: Dataset({
        features: ['input', 'target'],
        num_rows: 7056
    })
})


In [6]:
# Chia tập train/test/valid
# Note: df_valid dùng để thử các kết quả model trước khi áp dụng lần cuối với test
df_train = pd.DataFrame(dataset['train'])
df_test = pd.DataFrame(dataset['test'])
df_valid = pd.DataFrame(dataset['validation'])

In [4]:
df_train

,input,target
0,zinredie zidne: htlv có lương befo nhkấtl lịc ...,zinedine zidane: hlv có lương bèo nhất lịch sử...
1,Còn cứ kéo dài như vậy sẽ ko ổn.,Còn cứ kéo dài như vậy sẽ không ổn.
2,PVC trien khai nhieu cong trinh xay dung lon.,PVC triển khai nhiều công trình xây dựng lớn.
3,"""bữa cơm thwuwowrg khnôg có gì cao nơưn mỹ vị,...","""bữa cơm thưởng không có gì cao lương mỹ vị, c..."
4,"Dieu tra vu vo hui hang ti dong o Nam Can, Ca ...","Điều tra vụ vỡ hụi hàng tỉ đồng ở Năm Căn, Cà ..."
...,...,...
56440,May mà được ông làm tóc có tâm. Đến tết mái có...,May mà được ông làm tóc có tâm. Đến tết mái có...
56441,Nguoi dep Philippines dang quang HH The gioi 2...,Người đẹp Philippines đăng quang HH Thế giới 2...
56442,smartfone - người bn đồng hànk.,smartphone - người bạn đồng hành.
56443,"trongv vụ xử edan, cknúg ta cos lúg túng....","trong vụ xử vedan, chúng ta có lúng túng...."


### load and create vocabulary

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Tạo 1 vocab dựa trên 1 dataset khác
vocab = []
with open(r"/content/drive/MyDrive/vocabulary.txt", 'r', encoding='utf-8') as f:
    vocab = f.read().splitlines()

# Lọc các từ giống nhau
vocab = list(dict.fromkeys(vocab))

# Ánh xạ word và idx và ngược lại
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for i, word in enumerate(vocab)}

In [3]:
# Tạo 1 vocab dựa trên 1 dataset khác
vocab = []
with open(r"D:\NLP\project\vocabulary.txt", 'r', encoding='utf-8') as f:
    vocab = f.read().splitlines()

# Lọc các từ giống nhau
vocab = list(dict.fromkeys(vocab))

# Ánh xạ word và idx và ngược lại
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for i, word in enumerate(vocab)}

In [6]:
print(len(vocab))
print(vocab[:10])

77814
['a', 'abagiua', 'abatoa', 'a_bàng', 'a_bung', 'a_di', 'a_di_đà_kinh', 'adiđà_phật', 'a_di_đà_phật', 'a_di_đà_tam_tôn']


Tạo 1 Dictionary là counts để tính số lần xuất hiện của mỗi từ trong vocab\
Lợi ích: sau này ta sẽ cộng điểm ưu tiên cho thằng có số lần xuất hiện lớn hơn\

In [7]:
counts = defaultdict(int)

for sentence in df_train['target']:
    sentence = sentence.lower()
    words = sentence.split()

    for i in range(len(words)):
        word = words[i]
        counts[word] += 1

# Các thuật toán sửa lỗi sẽ sử dụng:
(Note: thuật toán sửa lỗi 1, 2 là ở trong Symspell Correction, mình tách ra code riêng chứ k xài hàm có sẵn)

### Thuật toán 1: Sửa lỗi chính tả của từ
Sử dụng Symmetric Delete:
Chỉ sử dụng phép Xóa (Delete) để thu hẹp không gian tìm kiếm.\
Nếu hai từ có khoảng cách chỉnh sửa (Edit Distance) là $k$, chúng sẽ có chung ít nhất một biến thể "xóa đi $n$ ký tự" ($n \le k$).

Import file từ 1 chữ tiếng Việt -> nguyên âm - phụ âm - dấu câu trong telex\
VD: 'ắ' - 'a', 'w', 's'

In [8]:
with open(r"D:\NLP\project\telex.txt", "r", encoding="utf-8") as f:
    telex = f.read()

telex = re.sub(r',\s*}', '\n}', telex)
telex = json.loads(telex)
telex

{'ă': ['a', 'w', ''],
 'â': ['a', 'a', ''],
 'ê': ['e', 'e', ''],
 'ô': ['o', 'o', ''],
 'ơ': ['o', 'w', ''],
 'ư': ['u', 'w', ''],
 'đ': ['d', 'd', ''],
 'á': ['a', '', 's'],
 'à': ['a', '', 'f'],
 'ã': ['a', '', 'x'],
 'ả': ['a', '', 'r'],
 'ạ': ['a', '', 'j'],
 'ắ': ['a', 'w', 's'],
 'ằ': ['a', 'w', 'f'],
 'ẵ': ['a', 'w', 'x'],
 'ẳ': ['a', 'w', 'r'],
 'ặ': ['a', 'w', 'j'],
 'ấ': ['a', 'a', 's'],
 'ầ': ['a', 'a', 'f'],
 'ẫ': ['a', 'a', 'x'],
 'ẩ': ['a', 'a', 'r'],
 'ậ': ['a', 'a', 'j'],
 'é': ['e', '', 's'],
 'è': ['e', '', 'f'],
 'ẽ': ['e', '', 'x'],
 'ẻ': ['e', '', 'r'],
 'ẹ': ['e', '', 'j'],
 'ế': ['e', 'e', 's'],
 'ề': ['e', 'e', 'f'],
 'ễ': ['e', 'e', 'x'],
 'ể': ['e', 'e', 'r'],
 'ệ': ['e', 'e', 'j'],
 'í': ['i', '', 's'],
 'ì': ['i', '', 'f'],
 'ĩ': ['i', '', 'x'],
 'ỉ': ['i', '', 'r'],
 'ị': ['i', '', 'j'],
 'ó': ['o', '', 's'],
 'ò': ['o', '', 'f'],
 'õ': ['o', '', 'x'],
 'ỏ': ['o', '', 'r'],
 'ọ': ['o', '', 'j'],
 'ố': ['o', 'o', 's'],
 'ồ': ['o', 'o', 'f'],
 'ỗ': ['o', 'o'

Thử nghiệm hàm

In [9]:
def create_telex_form(word):
    word = word.lower()
    prefix = ""      # Phụ âm đầu
    vowel_base = ""  # Nguyên âm gốc
    suffix = ""      # Phụ âm cuối
    word_tone = ""   # Dấu thanh
    word_mod = ""    # Ký tự gõ mũ/móc
    
    VOWELS = "aeiouy" # Các nguyên âm tiếng Việt
    state = 0 # 0: phụ âm đầu, 1: nguyên âm
    
    i = 0
    while i < len(word):
        step = 1
        # Trường hợp 'ươ'
        if i < len(word) - 1 and word[i:i+2] in telex:
            char = word[i:i+2]
            step = 2
        else:
            char = word[i]
            
        # Nếu ký tự nằm trong từ điển
        if char in telex:
            if char == 'đ':
                if state == 0: prefix += 'dd'
                else: suffix += 'dd'
            else:
                vowel_base += telex[char][0]
                if telex[char][1]: word_mod = telex[char][1]
                if telex[char][2]: word_tone = telex[char][2]
                state = 1
                
        # Nếu ký tự là chữ bình thường
        else:
            if char in VOWELS:
                vowel_base += char
                state = 1
            else:
                if state == 0: prefix += char
                else: suffix += char
                
        i += step

    # Tạo biến thể
    variants = set()
    inline_vowel = vowel_base + word_mod
    
    # Kiểu 1 & 2: Gõ chuẩn ngay sau nguyên âm hoặc ném dấu thanh ra cuối
    variants.add(prefix + inline_vowel + word_tone + suffix)
    variants.add(prefix + inline_vowel + suffix + word_tone)
    
    # Kiểu 3: Phím bổ nghĩa (w, a, e, o) ném ra cuối từ
    if word_mod:
        variants.add(prefix + vowel_base + word_tone + suffix + word_mod)
        variants.add(prefix + vowel_base + suffix + word_mod + word_tone)
        variants.add(prefix + vowel_base + suffix + word_tone + word_mod)
        
    # Trường hợp đặc biệt của "ươ"
    if vowel_base == 'uo' and word_mod == 'w':
        # Tách w hai lần ngay sau nguyên âm
        variants.add(prefix + 'uwow' + word_tone + suffix)
        variants.add(prefix + 'uwow' + suffix + word_tone)
        
        # Tách w đầu, w cuối
        variants.add(prefix + 'uwo' + word_tone + suffix + 'w')
        variants.add(prefix + 'uwo' + suffix + 'w' + word_tone)
        variants.add(prefix + 'uwo' + suffix + word_tone + 'w')
        
    return list(v for v in variants if v)

In [26]:
create_telex_form("trường")

['truongwf',
 'truongfw',
 'truwowfng',
 'truwofngw',
 'truowngf',
 'truwongfw',
 'truwongwf',
 'truowfng',
 'truwowngf',
 'truofngw']

Hàm tạo các biến thể xóa từ 0 đến k kí tự của từ, hàm trả về list các biến thể xóa

In [10]:
def get_deletes(word, k = 2):
    queue = [word]
    variant_list = set()

    for _ in range(k):
        temp_queue = set()
        for w in queue:
            if len(w) > 1:
                for i in range(len(w)):
                    variant = w[:i] + w[i+1:]
                    variant_list.add(variant)
                    temp_queue.add(variant)
        queue = temp_queue
    return variant_list

In [50]:
# Kiểm tra hàm
print(get_deletes("chào"))

{'hào', 'hà', 'co', 'chà', 'ho', 'ào', 'cà', 'cho', 'ch', 'cào'}


In [11]:
# Tạo danh sách các từ thiếu từ 0 đến k kí tự trong vocab
# Tạo 1 Dictionary (Hash Map) để lưu toàn bộ biến thể xóa
sym_dict = defaultdict(list)

for word in vocab:
    # Lấy các biến thể và từ gốc của 1 từ
    base_forms = [word] + create_telex_form(word)

    for form in base_forms:
        # Lưu form này (distance 0)
        if word not in sym_dict[form]:
            sym_dict[word].append(word)
        
    # Tạo deletes cho từng form và lưu vào từ điển
    variant_list = get_deletes(form)
    for variant in variant_list:
        # Map biến thể tới từ gốc 'word' hiện tại nếu chưa map
        if word not in sym_dict[variant]:
            sym_dict[variant].append(word)

In [53]:
# Kiểm tra
sym_dict

defaultdict(list,
            {'a': ['a',
              'aga',
              'ala',
              'ami',
              'avi',
              'à',
              'ả',
              'á',
              'ạ',
              'abc',
              'ác',
              'adn',
              'ag',
              'ai',
              'ải',
              'ái',
              'ak',
              'al',
              'am',
              'ám',
              'an',
              'án',
              'ang',
              'anh',
              'ao',
              'ào',
              'ảo',
              'áo',
              'áp',
              'ar',
              'as',
              'át',
              'atk',
              'au',
              'áy',
              'ăă',
              'ă',
              'ăn',
              'â',
              'âm',
              'ân',
              'âu',
              'ba',
              'bà',
              'bả',
              'bã',
              'bá',
              'bạ',
              '

In [12]:
# Hàm tính khoảng cách của xâu 1 và xâu 2 bằng thuật toán Damerau-Levenshtein
# Là hàm edit_distance nhưng có thêm phép đổi chỗ các kí tự (gõ lộn thứ tự)
# Cải thiện thêm bằng khoảng cách bàn phím cho trường hợp gõ nhầm

# Các phím liền kề trên bàn phím để tính trọng số
ADJACENT_KEYS = {
    'q': 'wea', 'w': 'qeasd', 'e': 'wrsdf', 'r': 'etdfg', 't': 'ryfgh', 'y': 'tughj', 'u': 'yihjk', 'i': 'uojkl', 'o': 'ipkl', 'p': 'ol',
    'a': 'qwsz', 's': 'weadzx', 'd': 'ersfxc', 'f': 'rtdgcv', 'g': 'tyfhvb', 'h': 'yugjbn', 'j': 'uihknm', 'k': 'iojlm', 'l': 'opk',
    'z': 'asx', 'x': 'sdzc', 'c': 'dfxv', 'v': 'fgcb', 'b': 'ghvn', 'n': 'hjbm', 'm': 'jkn'
}

# Các phím mang tính năng của Telex
TELEX_KEYS = set(['s', 'f', 'r', 'x', 'j', 'w', 'a', 'e', 'o', 'd'])

def edit_distance(s1, s2):
    n, m = len(s1), len(s2)
    dp = [[0.0] * (m + 1) for _ in range(n + 1)]

    # Khởi tạo giá trị hàng và cột đầu tiên
    for i in range(n + 1):
        dp[i][0] = i

    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            char1 = s1[i - 1]
            char2 = s2[j - 1]
            # Tính chi phí thay thế (0 nếu giống nhau, 0.5 nếu khác mà gần nhau trên bàn phím, 1 nếu khác và ở xa trên bàn phím)
            if char1 == char2:
                sub_cost = 0.0
            elif char1 in ADJACENT_KEYS.get(char2, "") or char2 in ADJACENT_KEYS.get(char1, ""):
                # Nếu gõ nhầm 2 phím cạnh nhau (VD: a và s), chi phí chỉ là 0.5
                sub_cost = 0.5 
            else:
                # Lỗi gõ nhầm phím xa nhau, chi phí 1.0
                sub_cost = 1.0

            # Tính chi phí thêm / xóa
            del_cost = 0.8 if char1 in TELEX_KEYS else 1.0
            ins_cost = 0.8 if char2 in TELEX_KEYS else 1.0

            dp[i][j] = min(
                dp[i - 1][j] + del_cost,       # Xóa
                dp[i][j - 1] + ins_cost,       # Thêm
                dp[i - 1][j - 1] + sub_cost    # Thay thế
            )

            # Phép Đổi chỗ (Transposition)
            if i > 1 and j > 1 and s1[i - 1] == s2[j - 2] and s1[i - 2] == s2[j - 1]:
                dp[i][j] = min(dp[i][j], dp[i - 2][j - 2] + 0.5)

    return dp[n][m]

In [39]:
# Kiểm tra hàm
edit_distance("truwowfng", "trwuwowfng")

0.8

In [16]:
# Hàm tìm từ gần nhất với từ nhập vào, hàm trả về các từ gần nhất và khoảng cách của nó
def lookup(word, k=3):
    # Trả về tuple (từ, khoảng cách) để đồng nhất với kết quả ở dưới
    if word in vocab:
        return [(word, 0)]

    variant_list = get_deletes(word)
    variant_list.add(word)

    # Lưu đáp án là các từ gần nhất và khoảng cách của nó
    candidates = {}

    # Tra cứu các biến thể
    for variant in variant_list:
        if variant in sym_dict:
            for suggestion in sym_dict[variant]:
                if suggestion in candidates:
                    continue

                # SỬ DỤNG HÀM TELEX ĐỂ TÍNH KHOẢNG CÁCH (NẾU BẠN KIỂM TRA LỖI GÕ PHÍM)
                telex_form = [suggestion] + create_telex_form(suggestion)

                dist = min([edit_distance(word, form) for form in telex_form])

                # Nhận khi khoảng cách <= k, tồn tại trong từ điển và có tần suất > 0
                if dist <= k and suggestion in vocab and counts.get(suggestion, 0) > 0:
                    candidates[suggestion] = (dist, counts.get(suggestion, 0))

    # Xếp hạng ưu tiên: Distance nhỏ trước -> Tần suất cao trước
    result = sorted(candidates.items(),
                    key=lambda x: (x[1][0], -x[1][1]))

    ans = []
    # Giải nén thẳng tuple cho dễ đọc, đổi tên biến tránh ghi đè tham số 'word'
    for cand_word, (dist, count) in result:
        # Sử dụng tham số k thay vì hardcode số 2
        if dist > k: 
            break
        
        # Trả về cả từ và khoảng cách 
        ans.append((cand_word, dist))
        
    return ans

In [18]:
# Kiểm tra hàm
lookup("trwuwowfng")

[('trường', 0.8)]

### Thuật toán 2: viết tắt (Teencode & Informal Variants)
Mapping (tạo các cặp viết tắt và từ chính) + Contextual Ranking (xem thử cái nào khả thi hơn)\
Kết hợp Deep Learning để xem xét ngữ cảnh của từ viết tắt từ đó tăng độ chính xác

### Thuật toán 3: Sửa lỗi viết không dấu
2 cách:
- 1 là N-gram + Viterbi
- 2 là model BARTpho-syllable nhỏ

### Model chung: Chọn từ phù hợp với ngữ cảnh
Dựa trên Skip-gram (window_size = 5) + log-Probability\
Hướng sử dụng là check tìm trong câu các từ không có trong từ điển -> gán lỗi cho nó -> sử dụng thuật toán sửa lỗi 1 tìm ra top n từ có thể sẽ đúng (ở đây n = 5) -> xem thử có trong từ điển không -> xem từ nào gần nhất với các từ trong câu\
VD: hôm nay tôi đi hõc ở trường
- hõc không có trong từ điển -> sai -> các từ thay thế: học, hóc, húc, hạc, hắc
- tính điểm dựa trên các yếu tố sau và chọn thằng có điểm cao nhất
    - 1. Điểm dựa vào ngữ cảnh (skip-gram trừ stopwords)
    - 2. Điểm dựa vào tần xuất (n-gram)
    - 3. Điểm cộng nếu tạo được thành 1 từ có trong từ điển
    - 4. Điểm trừ nếu ứng cử viên là 1 từ trong stopword

Load danh sách stopwords

In [24]:
with open(r"D:\NLP\project\vietnamese-stopwords.txt", 'r', encoding='utf-8') as f:
    stopwords = f.read().splitlines()
stopword = set(stopwords)

Tạo các cặp skipgram với window_size = 5

In [25]:
window_size = 5
skipgram_data = []

for sentence in df_train['target']:
    # Lấy danh sách từ
    sentence = sentence.lower()
    words = sentence.split()

    # Lọc stopword
    filtered_words = []
    for word in words:
        if word not in word_to_idx or word in stopword:
            continue

        filtered_words.append(word)

    for i, word in enumerate(filtered_words):
        # Kiểm tra xem từ này có trong vocab
        if word not in word_to_idx:
            continue

        center = word_to_idx[word]

        for j in range(-window_size, window_size + 1):
            # Kiểm tra các khoảng cách xem có phù hợp (trừ chính nó)
            if j == 0:
                continue
            if 0 <= i + j < len(filtered_words):
                context_word = words[i + j]
                #Kiểm tra các từ bên cạnh có trong vocab không
                if context_word in word_to_idx:
                    context = word_to_idx[context_word]
                    skipgram_data.append((center, context))

In [26]:
len(skipgram_data)

1996340

model Skipgram

In [27]:
class SkipGram(nn.Module):

    def __init__(self, vocab_size, embed_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):

        embed = self.embedding(x)
        out = self.linear(embed)

        return out

In [ ]:
#  Chuẩn bị chạy bằng GPU
device = torch.device("cuda")

centers = torch.LongTensor([pair[0] for pair in skipgram_data])
contexts = torch.LongTensor([pair[1] for pair in skipgram_data])

batch_size = 4096
dataset = TensorDataset(centers, contexts)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
EMBED_DIM = 256
VOCAB_SIZE = len(vocab)
model_skipgram = SkipGram(VOCAB_SIZE, EMBED_DIM).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_skipgram.parameters(), lr=0.001)

epochs = 30

model_skipgram.train()

for epoch in range(epochs):
    total_loss = 0

    for batch_centers, batch_contexts in loader:

        batch_centers = batch_centers.to(device)
        batch_contexts = batch_contexts.to(device)

        # Forward
        outputs = model_skipgram(batch_centers)
        loss = criterion(outputs, batch_contexts)

        # Backward
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 3827.7262
Epoch 2, Loss: 3245.9217
Epoch 3, Loss: 3199.7056
Epoch 4, Loss: 3174.3389
Epoch 5, Loss: 3157.1457
Epoch 6, Loss: 3144.0318
Epoch 7, Loss: 3133.5485
Epoch 8, Loss: 3124.6086
Epoch 9, Loss: 3116.9301
Epoch 10, Loss: 3110.2487
Epoch 11, Loss: 3104.1077
Epoch 12, Loss: 3098.6840
Epoch 13, Loss: 3093.6828
Epoch 14, Loss: 3089.0322
Epoch 15, Loss: 3084.6882
Epoch 16, Loss: 3080.6714
Epoch 17, Loss: 3076.9367
Epoch 18, Loss: 3073.3004
Epoch 19, Loss: 3070.0118
Epoch 20, Loss: 3066.7181
Epoch 21, Loss: 3063.7396
Epoch 22, Loss: 3060.7959
Epoch 23, Loss: 3058.0509
Epoch 24, Loss: 3055.3211
Epoch 25, Loss: 3052.7774
Epoch 26, Loss: 3050.3782
Epoch 27, Loss: 3048.0014
Epoch 28, Loss: 3045.6947
Epoch 29, Loss: 3043.4060
Epoch 30, Loss: 3041.3871


Lưu model train skip-gram

In [ ]:
torch.save(model_skipgram, r'D:/NLP/project/model_skipgram.pth')

In [28]:
model_skipgram = torch.load(r'D:/NLP/project/model_skipgram.pth', weights_only=False)

In [ ]:
torch.save(model_skipgram, 'model_skipgram.pth')
from google.colab import files

files.download('model_skipgram.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Hàm tính độ liên quan của 2 từ

In [29]:
def calculate_similarity(word1, word2, embedding_matrix, word_to_idx):
    # Lấy vector của từng từ
    idx1 = word_to_idx[word1]
    idx2 = word_to_idx[word2]
    vec1 = embedding_matrix[idx1]
    vec2 = embedding_matrix[idx2]

    # Tính Cosine Similarity
    # Công thức: (A . B) / (||A|| * ||B||)
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    similarity = dot_product / (norm1 * norm2)
    return similarity

Test thử với các từ "hóc xương", "học xương". "hóc xương" sẽ phải cho ra điểm cao hơn so với "học xương"

In [30]:
embeddings = model_skipgram.embedding.weight.data
embedding_matrix = embeddings.cpu().numpy()

score1 = calculate_similarity("hóc", "xương", embedding_matrix, word_to_idx)
score2 = calculate_similarity("học", "xương", embedding_matrix, word_to_idx)

print(score1, score2)

0.06637567 0.019825403


Test thử với các từ "sóng thần", "sóng tiền". "sóng thần" sẽ phải cho ra điểm cao hơn so với "sóng tiền"

In [31]:
embeddings = model_skipgram.embedding.weight.data
embedding_matrix = embeddings.cpu().numpy()

score1 = calculate_similarity("sóng", "thần", embedding_matrix, word_to_idx)
score2 = calculate_similarity("sóng", "tiền", embedding_matrix, word_to_idx)

print(score1, score2)

0.14927007 0.05731002


Chuẩn bị để chuẩn hóa điểm count về [0, 1] cho dễ tính toán

In [32]:
all_log_counts = [math.log(c) for c in counts.values()]
min_log = min(all_log_counts)
max_log = max(all_log_counts)

def get_normalized_ngram(word, counts, min_log, max_log):
    c = counts.get(word, 1)
    log_c = math.log(c)
    # Ép về khoảng [0, 1]
    return (log_c - min_log) / (max_log - min_log + 1e-9)

Hàm chọn từ sửa sai dựa trên độ liên quan

In [33]:
def choose_best_candidate(candidates, sentence, error_idx, embedding_matrix, error_indices, 
                           word_to_idx, stopwords, counts, min_log, max_log, window_size=2):
    best_word = None
    max_total_score = -float('inf')

    # Chọn các từ để candidate xét, bỏ chính nó, các từ stopwords
    start = max(0, error_idx - window_size)
    end = min(len(sentence), error_idx + window_size + 1)
    
    context_words = [sentence[i] for i in range(start, end) 
                     if i not in error_indices and sentence[i] not in stopwords]

    for candidate in candidates:
        # 1. Tính điểm dựa trên ngữ cảnh (chuẩn hóa lại điểm từ 0 đến 1)
        sim_sum = 0
        valid_contexts = 0
        for context in context_words:
            if context in word_to_idx:
                sim = calculate_similarity(candidate, context, embedding_matrix, word_to_idx)
                sim_sum += max(0, sim) # Chỉ lấy điểm dương
                valid_contexts += 1
        
        avg_sim = sim_sum / valid_contexts if valid_contexts > 0 else 0

        # 2. Tính điểm tần suất (chuẩn hóa lại điểm từ 0 đến 1)
        log_c = math.log(counts.get(candidate, 1))
        norm_ngram = (log_c - min_log) / (max_log - min_log + 1e-9)

        # 3. Kiểm tra xem có tạo từ ghép tồn tại trong từ điển (điểm thưởng)
        bigram_bonus = 0
        
        # Kiểm tra với từ đứng trước
        if error_idx > 0:
            prev_word = sentence[error_idx - 1]
            compound_prev = f"{prev_word}_{candidate}"
            if compound_prev in word_to_idx:
                bigram_bonus += 2.0 # Cộng một điểm thưởng rất lớn

        # Kiểm tra với từ đứng sau
        if error_idx < len(sentence) - 1:
            next_word = sentence[error_idx + 1]
            compound_next = f"{candidate}_{next_word}"
            if compound_next in word_to_idx:
                bigram_bonus += 2.0
        
        # 4. Trừ điểm nếu candidate là stopword
        # penalty = 1.5 if candidate in stopwords else 0
        penalty = 0.02

        # Trộn điểm với trọng số
        # Alpha=0.7 (Tin vào ngữ cảnh), Beta=0.3 (Tin vào tần suất)
        total_score = (0.5 * avg_sim) + (0.2 * norm_ngram) + (0.3 * bigram_bonus) - penalty

        if total_score > max_total_score:
            max_total_score = total_score
            best_word = candidate
        print(candidate, total_score)

    return best_word if best_word is not None else candidates[0]

Tạo hàm để thực hiện thuật toán

In [34]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s_]', '', text)
    text = text.split()
    return text

In [35]:
def correct_spelling_errors(sentence, embedding_matrix, word_to_idx, stopwords, counts):
    sentence = preprocess(sentence)
    error = []

    # Tìm vị trí các từ bị lỗi
    for i, word in enumerate(sentence):
        if word in word_to_idx:
            continue

        candidates = lookup(word)
        if len(candidates) == 0:
            continue
        
        error.append(i)
        
    # Bắt đầu quá trình sửa lỗi
    for error_idx in error:
        candidates = lookup(sentence[error_idx])
        
        # Gọi hàm và truyền đầy đủ các tham số cần thiết
        if candidates:
            sentence[error_idx] = choose_best_candidate(
                candidates=candidates, 
                sentence=sentence, 
                error_idx=error_idx, 
                embedding_matrix=embedding_matrix, 
                error_indices=error,
                word_to_idx=word_to_idx,
                stopwords=stopwords,
                counts=counts,
                min_log=min_log,
                max_log=max_log
            )

    # Dùng join để ghép chuỗi chuẩn xác và tối ưu hiệu năng
    sentence = " ".join(sentence)
    return sentence

In [64]:
embeddings = model_skipgram.embedding.weight.data
embedding_matrix = embeddings.cpu().numpy()

# Test sửa lỗi
sentence = "Hôm nay tôi đi hõc ở trường."
fixed_sentence = correct_spelling_errors(
    sentence=sentence, 
    embedding_matrix=embedding_matrix, 
    word_to_idx=word_to_idx, 
    stopwords=stopwords, 
    counts=counts
)

print(fixed_sentence)

học 0.75139534
hóc 0.05176587
húc 0.06701155
hạc 0.052439343
hắc 0.062902465
hốc 0.048587706
hực 0.029426832
hộc 0.033670776
hục 0.025936235
các 0.15993743
hay 0.14134675921989812
hơn 0.14249155
hà 0.16796875
hội 0.17153
hợp 0.16375926
hết 0.15630327
thực 0.16695717
khác 0.13435148
chắc 0.13659269
lúc 0.12786506
hai 0.15075938
thức 0.13189045
chức 0.12560312
tác 0.1324743
bác 0.12284508
lực 0.18759029
hoa 0.1428281
họ 0.13916585
hồ 0.13818094
phục 0.12649693
tục 0.14668496
hóa 0.12662055
sức 0.12791798
hệ 0.123036265
hôm 0.11695084
hải 0.1532506
hồi 0.16599144
đọc 0.13441731
hộ 0.12768337
hậu 0.15789178
cực 0.15082447
hỏi 0.14853577
mức 0.12803346
đặc 0.17663185
nhạc 0.12643719
mặc 0.13371031
đức 0.1474541
hài 0.119964436
hàn 0.12428433
chúc 0.13141502
bắc 0.1356535
độc 0.11886543
xác 0.13010372
vực 0.15633582
sắc 0.1140544
hơi 0.7288358
hả 0.10939324
hát 0.11096047
hoặc 0.10323364682642022
hạ 0.10290796335994072
mắc 0.10282578295805467
khóc 0.12893306
hại 0.114089414
phúc 0.14313287
t

In [66]:
sentence = "sóng thền thì dất nguy hiểm"
fixed_sentence = correct_spelling_errors(
    sentence=sentence, 
    embedding_matrix=embedding_matrix, 
    word_to_idx=word_to_idx, 
    stopwords=stopwords, 
    counts=counts
)

print(fixed_sentence)

tiền 0.16824931
thân 0.14574853
thần 0.7809552
thận 0.08477536
thôn 0.10095924
thề 0.12379718
than 0.05996590011687471
thon 0.07799794
thềm 0.062614724
then 0.09011096
thốn 0.019388663926666248
thẹn 0.036493238
thìn 0.03597183
thiền 0.038745463
thồn 0.007443629
thun 0.03503259
thì 0.21410142
thế 0.7471692084188913
trên 0.1483948
thành 0.16863877
thấy 0.1415315530754361
thể 0.19271713
nhân 0.14056207940537171
hơn 0.13934554597369236
theo 0.17650174
thôi 0.7345803987741845
tin 0.13638741
nhận 0.13286445258868812
thời 0.13120784585787307
thực 0.14342082
thật 0.15187743
thông 0.15121911
thanh 0.12856475871409112
thủ 0.1720789
nhìn 0.12765246050565057
toàn 0.13768227
thị 0.1270863374117468
thi 0.14057852
thằng 0.16323395
thầy 0.15493505
trần 0.16111316
tháng 0.2071095
thu 0.79935336
thức 0.13961476
tiên 0.16227067
thứ 0.14215575
thêm 0.17210367
phần 0.12621866
thống 0.18010779
tham 0.11639337
tên 0.11511618142431722
thái 0.11468838611451206
thay 0.1356363
trận 0.15005131
tiến 0.16030623
thắ

In [ ]:
sentence = "ít nhất 5 người chết do sóng thnầ tạj xloomon."
fixed_sentence = correct_spelling_errors(
    sentence=sentence, 
    embedding_matrix=embedding_matrix, 
    word_to_idx=word_to_idx, 
    stopwords=stopwords, 
    counts=counts
)

print(fixed_sentence)

thần 0.7809552
thì 0.21410142
thế 0.14716920841889136
thành 0.16863877
thấy 0.1415315530754361
tại 0.21812934
tạo 0.11661717101467366
tạm 0.124166414
tạ 0.09374051
tạp 0.07896966
ít nhất 5 người chết do sóng thần tại xloomon


In [67]:
sentence = "Tôi đang xửa lỗi bản hợp đồng này"
fixed_sentence = correct_spelling_errors(
    sentence=sentence, 
    embedding_matrix=embedding_matrix, 
    word_to_idx=word_to_idx, 
    stopwords=stopwords, 
    counts=counts
)

print(fixed_sentence)

tôi đang xửa lỗi bản hợp đồng này


# Kết hợp các thuật toán sửa lỗi
Biến các thuật toán thành 1 hàm sửa lỗi câu theo trường hợp
- 1. Sai lỗi chính tả  
- 2. Viết tắt
- 3. Viết không dấu     \
=> Tất cả sẽ dựa trên ngữ cảnh

# Test trên tập valid

Lấy top 1 câu sau chỉnh sửa để xem trên tập valid xem kết quả

# Chỉnh sửa phù hợp

Điều chỉnh thuật toán, dữ liệu, vv cho phù hợp lại

# Đánh giá lần cuối trên tập test